<a href="https://colab.research.google.com/github/KaterynaKon/irish-tender-value/blob/main/public_tender_value.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Estimating the Expected Contract Value of Irish Public Construction Tenders

## Phase 1 - Business Understanding

### Context and problem

Ireland's public sector buys construction work through competitive tenders: public bodies advertise their projects on the eTenders platform, and contractors submit priced bids by a deadline. A single bid requires weeks of estimator and quantity-surveyor time, while dozens of relevant notices may be published in a month, so a regional firm must decide where to invest that effort before the submission deadline.

Construction companies in Ireland lack a tool to estimate the expected contract value of public tenders before submitting a bid. This leads to wasted resources on low-value tenders and missed high-value opportunities.

"Expected contract value" is operationalised as the awarded value: the amount the winning bidder is contracted to be paid.

The project is limited to construction works contracts (CPV code 45), which covers building projects such as housing as well as civil engineering works.

### Stakeholders and users


The tool is used by bid and tender managers in Irish construction firms, who decide each week which published tenders to pursue. The data is supplied by contracting authorities and by the Office of Government Procurement; neither is asked to do anything additional for this project.

| Stakeholder | Role | Interest |
| --- | --- | --- |
| Bid / tender manager | Primary user | A ranked shortlist with an expected value, produced before the submission deadline |
| Estimating team | Prepares the priced bid | Fewer bids prepared for contracts that turn out to be marginal |
| Managing director / owner | Approves bid preparation spend | A predictable pipeline and a revenue forecast |
| Contracting authorities | Supply the data by publishing notices | Correct use of the published record; no additional reporting burden |
| Office of Government Procurement | Publishes the dataset under CC BY 4.0 | Attribution and correct representation of the data |

### Business or practical objective

The objective is to give the bid manager an expected contract value for every live construction tender before the submission deadline, so that estimating effort is directed towards contracts that are worth pursuing. In practice this supports one recurring decision - bid or no-bid - made every week, across dozens of notices, in minutes rather than hours.

Success at the business level means the firm stops preparing bids for contracts far below its target value, and stops overlooking contracts that are worth more than the notice alone suggests. The output is a ranked list with an expected value and a range, used as an input to the decision, not as the decision itself.


### Analytics objective
The task is regression: for each published construction tender, estimate in euros the value of the contract that will eventually be awarded, using only what the notice says. This turns the practical objective into something measurable — an estimate whose error can be checked against tenders that have already been awarded.

The scope is construction works contracts.

### Success criteria

Technical: the error must be at least one third lower than that of the baseline model (which forecasts the average contract value for every tender).

Practical: the top 20% of the ranking must capture at least 50% of the total contract value, and a typical prediction must be within 40% of the true value.

### Constraints and assumptions

Data: public open dataset only - no internal documents, bid prices or company cost data.

Awarded tenders only: the model predicts the value of a contract that is actually awarded, not of a competition that is later cancelled.

Money: values in euro as recorded in the award year; some may exclude VAT, and some may cover framework agreements with multiple call-offs.

Inflation: not adjusted for. The publication year is kept as a feature, so the model can learn each period's price level. Adjusting to constant prices is possible later.

## Phase 2: Data Understanding

### Initial Data Collection Report

| Item | Detail |
| --- | --- |
| Dataset | Public Procurement Opendata Dataset — Contract Notices Published on eTenders |
| Publisher | Office of Government Procurement (Ireland) |
| Source | https://data.gov.ie/dataset/contract-notices-published-on-etenders |
| Licence | Creative Commons Attribution 4.0 (CC BY 4.0) |
| Coverage | Competitions published 01/01/2013 – 30/09/2026 |
| Unit of analysis | One published tender competition |
| Accessed | 6 November 2026 |

Files acquired: one CSV file with 31 columns. No other tables are used and no joins are required.

### Data Description Report

In [7]:
import pandas as pd

df = pd.read_csv("sample_data/Public_Procurement_Opendata_Dataset.csv", encoding="cp1252", low_memory=False)
df.head()


,Tender ID,Parent Agreement ID,Contracting Authority,Name of Client Contracting Authority,Agreement Owner,Tender/Contract Name,Notice Published Date / Contract Created Date,Directive,Competition Type,Main Cpv Code,...,Award Published,Sum of Awarded Value (€),Sum of No of Bids Received,Sum of No of SMEs Bids Received,Awarded Suppliers,Sum of No of Awarded SMEs,TED Notice Link,TED CAN Link,Platform,Source
0,6671287.0,3502944.0,The Office of Government Procurement,An Garda Síochána,The Office of Government Procurement,IPS017F-01-004-PROJ000010668-SRFT for External...,30/06/2026,Classic,FW - Mini-Comp,72000000.0,...,30/06/2026,13470600.0,NaN,NaN,Ergo,0.0,NaN,NaN,ED Platform,OpenData-Contracts
1,7249646.0,4574076.0,The Office of Government Procurement,Houses of the Oireachtas Service,The Office of Government Procurement,ITC016F-01-009 SRFT for Network Products and A...,30/06/2026,Classic,FW - Mini-Comp,32410000.0,...,30/06/2026,1922671.0,NaN,NaN,Vodafone,0.0,NaN,NaN,ED Platform,OpenData-Contracts
2,7289488.0,2624317.0,The Office of Government Procurement,Dublin and Dun Laoghaire Education and Trainin...,The Office of Government Procurement,SRFT for Lift and Escalator Maintenance for Du...,30/06/2026,Classic,FW - Mini-Comp,42400000.0,...,30/06/2026,203015.0,NaN,NaN,Orona MID WESTERN LIFTS,1.0,NaN,NaN,ED Platform,OpenData-Contracts
3,7519112.0,1886457.0,Tipperary County Council,NaN,Tipperary County Council,Tipperary County Council - Refurbishment Works...,30/06/2026,Classic,FW - Mini-Comp,45000000.0,...,30/06/2026,28790.0,NaN,NaN,J and S Cahalan Builders LTD,1.0,NaN,NaN,ED Platform,OpenData-Contracts
4,7558528.0,1794066.0,Health Service Executive (HSE),NaN,The Office of Government Procurement,HSE 28286 - Provision of Business Management C...,30/06/2026,Classic,FW - Mini-Comp,72000000.0,...,25/06/2026,3019896.0,NaN,NaN,Grant Thornton,0.0,NaN,NaN,ED Platform,OpenData-Contracts


**Encoding**
 The dataset is distributed in cp1252, not UTF-8. Reading it with the default UTF-8 codec fails immediately on the header, because the euro sign € is byte 0x80 in cp1252, which is not a valid UTF-8 start byte. Consequence: all text fields must be read with an explicit encoding="cp1252";

 **low_memory=False** makes pandas read the file in one pass and infer column types once. For monetary columns, this guards against a number/text mix that would otherwise distort the error and top-20% value metrics.

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87427 entries, 0 to 87426
Data columns (total 31 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   Tender ID                                      83517 non-null  float64
 1   Parent Agreement ID                            3719 non-null   float64
 2   Contracting Authority                          87394 non-null  object 
 3   Name of Client Contracting Authority           9505 non-null   object 
 4   Agreement Owner                                3719 non-null   object 
 5   Tender/Contract Name                           87427 non-null  object 
 6   Notice Published Date / Contract Created Date  85884 non-null  object 
 7   Directive                                      31079 non-null  object 
 8   Competition Type                               87427 non-null  object 
 9   Main Cpv Code                                  356